<a href="https://colab.research.google.com/github/Botatopiece9/kpneumo-amr-prediction/blob/main/01_fetch_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/kpneumo-amr-prediction"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder ready:", PROJECT_DIR)

Project folder ready: /content/drive/MyDrive/kpneumo-amr-prediction


In [ ]:
BASE_URL = "https://www.bv-brc.org/api/genome_amr/"
query = "eq(taxon_id,573)&eq(antibiotic,meropenem)&limit(5)"

url = BASE_URL + "?" + query
print(url)

response = requests.get(url, headers={"Accept": "application/json"})
print(response.status_code)

https://www.bv-brc.org/api/genome_amr/?eq(taxon_id,573)&eq(antibiotic,meropenem)&limit(5)
200


In [ ]:
data = response.json()
print(type(data), len(data))

df = pd.DataFrame(data)
df.head()

<class 'list'> 5


,id,antibiotic,owner,genome_id,genome_name,computational_method_performance,date_inserted,taxon_id,date_modified,computational_method,evidence,resistant_phenotype,public,_version_,vendor
0,3fba8fc3-1b67-423c-8abc-30438f9e5ee0,meropenem,PATRIC@patricbrc.org,573.21864,Klebsiella pneumoniae strain EuSCAPE_IT384,"Accuracy:0.916, F1 score:0.922, AUC:0.949",2018-10-05T14:29:55.486Z,573,2018-10-05T14:29:55.486Z,AdaBoost Classifier,Computational Method,Resistant,True,1818811210877894700,NaN
1,3fbd6834-f91c-4851-a487-43093c8d2036,meropenem,PATRIC@patricbrc.org,573.20165,Klebsiella pneumoniae strain NCTC11358,"Accuracy:0.916, F1 score:0.922, AUC:0.949",2018-10-04T04:35:34.941Z,573,2018-10-04T04:35:34.941Z,AdaBoost Classifier,Computational Method,Susceptible,True,1818811210895720400,NaN
2,3fc173dc-c0e7-4081-9ee8-da1432744808,meropenem,BVBRC@patricbrc.org,573.66935,Klebsiella pneumoniae KP_NORM_BLD_2015_115991,"Accuracy:0.916, F1 score:0.922, AUC:0.949",2024-03-03T21:18:14.552Z,573,2024-03-03T21:18:14.552Z,AdaBoost Classifier,Computational Method,Susceptible,True,1818811210916692000,PATRIC
3,3fc41b0d-384e-443d-8ccc-48e3dd9f5162,meropenem,PATRIC@patricbrc.org,573.21101,Klebsiella pneumoniae strain EuSCAPE_BE104,"Accuracy:0.916, F1 score:0.922, AUC:0.949",2018-10-05T03:53:35.144Z,573,2018-10-05T03:53:35.144Z,AdaBoost Classifier,Computational Method,Susceptible,True,1818811210931372000,NaN
4,3fc526c2-2139-4f01-a2e3-b1f71e5cb178,meropenem,PATRIC@patricbrc.org,573.15488,Klebsiella pneumoniae strain 10315_6#79,"Accuracy:0.916, F1 score:0.922, AUC:0.949",2017-11-30T19:44:00.443Z,573,2017-11-30T19:44:00.443Z,AdaBoost Classifier,Computational Method,Susceptible,True,1818811210935566300,NaN


In [ ]:
df.columns.tolist()

['id',
 'antibiotic',
 'owner',
 'genome_id',
 'genome_name',
 'computational_method_performance',
 'date_inserted',
 'taxon_id',
 'date_modified',
 'computational_method',
 'evidence',
 'resistant_phenotype',
 'public',
 '_version_',
 'vendor']

In [ ]:
df["evidence"].value_counts()

,count
evidence,
Computational Method,5


In [ ]:
query = (
    "eq(taxon_id,573)"
    "&eq(antibiotic,meropenem)"
    "&eq(evidence,Laboratory%20Method)"
    "&limit(5)"
)

url = BASE_URL + "?" + query
response = requests.get(url, headers={"Accept": "application/json"})
print(response.status_code)

df = pd.DataFrame(response.json())
print(df["evidence"].value_counts())
print(df.columns.tolist())
print("Total matching records:", response.headers.get("Content-Range"))

200
evidence
Laboratory Method    5
Name: count, dtype: int64
['id', 'measurement_sign', 'antibiotic', 'pmid', 'measurement_unit', 'laboratory_typing_platform', 'testing_standard', 'owner', 'genome_name', 'genome_id', 'measurement', 'date_inserted', 'laboratory_typing_method', 'date_modified', 'measurement_value', 'taxon_id', 'testing_standard_year', 'evidence', 'public', 'resistant_phenotype', '_version_']
Total matching records: items 0-5/6225


In [ ]:
content_range = response.headers.get("Content-Range")
total = int(content_range.split("/")[-1])
print(total)

6225


In [ ]:
import time

PAGE_SIZE = 1000
all_records = []

for offset in range(0, total, PAGE_SIZE):
    query = (
        "eq(taxon_id,573)"
        "&eq(antibiotic,meropenem)"
        "&eq(evidence,Laboratory%20Method)"
        f"&limit({PAGE_SIZE},{offset})"
    )
    url = BASE_URL + "?" + query
    response = requests.get(url, headers={"Accept": "application/json"})
    response.raise_for_status()

    page = response.json()
    all_records.extend(page)
    print(f"Got records {offset} to {offset + len(page)}")

    time.sleep(1)

print("Total downloaded:", len(all_records))

Got records 0 to 1000
Got records 1000 to 2000
Got records 2000 to 3000
Got records 3000 to 4000
Got records 4000 to 5000
Got records 5000 to 6000
Got records 6000 to 6225
Total downloaded: 6225


In [ ]:
amr = pd.DataFrame(all_records)
print(amr.shape)

amr.to_csv(f"{PROJECT_DIR}/bvbrc_meropenem_lab_raw.csv", index=False)
print("Saved!")

(6225, 22)
Saved!


In [ ]:
print(amr["resistant_phenotype"].value_counts())
print("Unique genomes:", amr["genome_id"].nunique())

resistant_phenotype
Susceptible     2557
Resistant       1496
Intermediate     159
Name: count, dtype: int64
Unique genomes: 6074


In [ ]:
print("Total rows:", len(amr))
print("Missing phenotype:", amr["resistant_phenotype"].isna().sum())

Total rows: 6225
Missing phenotype: 2013


In [ ]:
cols = ["genome_id", "measurement_sign", "measurement_value",
        "measurement_unit", "laboratory_typing_method", "testing_standard"]

missing = amr[amr["resistant_phenotype"].isna()]
missing[cols].head(10)

,genome_id,measurement_sign,measurement_value,measurement_unit,laboratory_typing_method,testing_standard
3731,573.21114,,1,mg/L,Broth dilution,CLSI
3732,573.20953,<=,0.06,mg/L,Broth dilution,CLSI
3733,573.21863,<=,0.06,mg/L,Broth dilution,CLSI
3734,573.20772,>,32,mg/L,Broth dilution,CLSI
3735,573.65979,<=,0.5,mg/L,MIC,CLSI
3736,573.20960,,16,mg/L,Broth dilution,CLSI
3737,573.21311,<=,0.06,mg/L,Broth dilution,CLSI
3738,573.21601,<=,0.06,mg/L,Broth dilution,CLSI
3739,573.20958,<=,0.06,mg/L,Broth dilution,CLSI
3740,573.21557,<=,0.06,mg/L,Broth dilution,CLSI


In [ ]:
counts = amr["genome_id"].value_counts()
repeated_ids = counts[counts > 1].index

repeats = amr[amr["genome_id"].isin(repeated_ids)]
repeats.sort_values("genome_id")[["genome_id", "resistant_phenotype"] + cols[1:]].head(20)

,genome_id,resistant_phenotype,measurement_sign,measurement_value,measurement_unit,laboratory_typing_method,testing_standard
2867,573.15567,Susceptible,,,NaN,Broth dilution,CLSI
737,573.15567,Susceptible,,,NaN,Broth dilution,CLSI
3175,573.15580,Susceptible,,,NaN,Broth dilution,CLSI
2237,573.15580,Susceptible,,,NaN,Broth dilution,CLSI
2839,573.15592,Susceptible,,,NaN,Broth dilution,CLSI
2278,573.15592,Susceptible,,,NaN,Broth dilution,CLSI
332,573.15595,Susceptible,,,NaN,Broth dilution,CLSI
436,573.15595,Susceptible,,,NaN,Broth dilution,CLSI
2245,573.15602,Resistant,,,NaN,Broth dilution,CLSI
2979,573.15602,Resistant,,,NaN,Broth dilution,CLSI


In [ ]:
amr["measurement_value"] = pd.to_numeric(amr["measurement_value"], errors="coerce")

print(amr["measurement_unit"].value_counts(dropna=False))
print(amr["measurement_sign"].value_counts(dropna=False))
print(amr["testing_standard"].value_counts(dropna=False))
print("Rows with an MIC value:", amr["measurement_value"].notna().sum())

measurement_unit
mg/L    4667
NaN     1552
mm         6
Name: count, dtype: int64
measurement_sign
      2626
<=    2078
>      929
>=     432
=      151
<        9
Name: count, dtype: int64
testing_standard
CLSI           4060
EUCAST         1541
NaN             620
EUCAST,CLSI       4
Name: count, dtype: int64
Rows with an MIC value: 4673


In [ ]:
amr.loc[amr["measurement_unit"] != "mg/L", "measurement_value"] = None
print("Rows with a usable MIC:", amr["measurement_value"].notna().sum())

Rows with a usable MIC: 4667


In [ ]:
def clsi_meropenem(row):
    sign = row["measurement_sign"]
    value = row["measurement_value"]

    if pd.isna(value):
        return None

    if sign in ["<=", "<"]:
        if value <= 1:
            return "Susceptible"
        return None

    if sign == ">":
        if value >= 2:
            return "Resistant"
        return None

    if sign == ">=":
        if value >= 4:
            return "Resistant"
        return None

    if value <= 1:
        return "Susceptible"
    elif value >= 4:
        return "Resistant"
    else:
        return "Intermediate"

In [ ]:
amr["phenotype_clsi"] = amr.apply(clsi_meropenem, axis=1)

print(amr["phenotype_clsi"].value_counts(dropna=False))
print(pd.crosstab(amr["resistant_phenotype"], amr["phenotype_clsi"]))

phenotype_clsi
Susceptible     2398
Resistant       2090
None            1559
Intermediate     178
Name: count, dtype: int64
phenotype_clsi       Intermediate  Resistant  Susceptible
resistant_phenotype                                      
Intermediate                   63         21            0
Resistant                       0       1285            0
Susceptible                     2          2         1280


In [ ]:
amr["label"] = amr["phenotype_clsi"].fillna(amr["resistant_phenotype"])
print(amr["label"].value_counts(dropna=False))

label
Susceptible     3671
Resistant       2301
Intermediate     253
Name: count, dtype: int64


In [ ]:
clean = amr[amr["label"].isin(["Resistant", "Susceptible"])]
print(clean.shape)

(5972, 24)


In [ ]:
labels_per_genome = clean.groupby("genome_id")["label"].nunique()
conflicting = labels_per_genome[labels_per_genome > 1].index
print("Genomes with conflicting labels:", len(conflicting))

clean = clean[~clean["genome_id"].isin(conflicting)]
clean = clean.drop_duplicates(subset="genome_id")
clean = clean[["genome_id", "genome_name", "label"]]

print(clean.shape)
print(clean["label"].value_counts())

Genomes with conflicting labels: 7
(5820, 3)
label
Susceptible    3625
Resistant      2195
Name: count, dtype: int64


In [ ]:
clean.to_csv(f"{PROJECT_DIR}/meropenem_labels_clean.csv", index=False)
print("Saved!")

Saved!
